# 텍스트 분류 복습

이 노트북은 05_text_classification의 전체 흐름을 문제로 복습합니다. Naive Bayes와 TF-IDF 기준선, LSTM 순차 분류, 멀티레이블 분류, TextCNN을 작은 예제로 직접 구현합니다. 총 29문제이며, 각 문제 아래의 빈 코드 셀에 풀이를 작성하세요.

- 문제는 쉬운 벡터화·분류 흐름에서 신경망 학습과 오류 분석으로 진행됩니다.
- 값 하나를 외우기보다 입력·출력 shape, 정답 형식, 데이터 누수, 평가 지표를 함께 확인하세요.
- 수업용 작은 데이터의 점수는 일반화 성능이 아닙니다. 결과가 나온 이유와 한계를 주석으로 설명해 보세요.

## 원본 노트북과 문제 연결

- 01_naive_bayes_classifier.ipynb: 문제 1~6
- 02_rnn_classifier.ipynb: 문제 7~16
- 03_multi_label_classification.ipynb: 문제 17~22
- 04_cnn_classifier.ipynb: 문제 23~28
- 네 모델을 비교하고 선택 근거를 정리하는 문제: 문제 29

각 문제는 이전 문제의 변수나 개념을 일부 사용할 수 있습니다. 셀을 위에서부터 순서대로 푸는 것을 권장합니다.

In [ ]:
import copy
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, hamming_loss
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MultiLabelBinarizer
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 1. Naive Bayes와 TF-IDF 기준선

이 절에서는 문서를 단어 빈도 또는 TF-IDF 벡터로 바꾸고, 확률 기반 Naive Bayes가 어떻게 클래스 점수를 만드는지 확인합니다.

## 문제 1. CountVectorizer로 문서-단어 행렬 만들기

다음 감성 문장을 단어 빈도 행렬로 바꾸세요.

    toy_texts = [
        '재미 감동 추천',
        '재미 유쾌 추천',
        '지루 실망 비추천',
        '지루 최악 비추천',
    ]
    toy_labels = np.array(['긍정', '긍정', '부정', '부정'])

조건:

- CountVectorizer를 만들고 fit_transform 결과를 toy_counts에 저장하세요.
- 학습된 어휘와 toy_counts의 shape를 출력하세요.
- 행과 열이 각각 무엇을 뜻하는지 주석으로 작성하세요.
- 단어 순서가 바뀌어도 같은 벡터가 되는 이유를 설명하세요.

## 문제 2. Multinomial Naive Bayes의 확률 예측

문제 1의 toy_counts와 toy_labels로 MultinomialNB를 학습하고 새 문장을 예측하세요.

    new_texts = ['재미 추천', '지루 비추천', '재미 지루']

조건:

- alpha=1.0으로 모델을 학습하세요.
- 새 문장은 기존 vectorizer의 transform으로 변환하세요.
- classes_, predict 결과, predict_proba 결과를 문장별로 출력하세요.
- 확률 행의 열 순서와 classes_의 관계를 주석으로 설명하세요.

## 문제 3. alpha 평활의 역할 비교

같은 학습 데이터를 사용해 alpha=0.1 모델과 alpha=10.0 모델을 각각 학습하세요.

조건:

- '재미 지루'의 클래스별 확률을 두 모델에서 비교하세요.
- 각 모델의 feature_log_prob_ 중 한 클래스 행을 출력하세요.
- alpha가 너무 작거나 너무 클 때 생길 수 있는 문제를 작성하세요.
- alpha는 학습 epoch 수가 아니라 확률 보정 강도라는 점을 주석으로 남기세요.

## 문제 4. TF-IDF의 fit과 transform 분리

다음 학습·테스트 문서를 TF-IDF 벡터로 바꾸세요.

    train_docs = ['space rocket orbit', 'hockey goal team', 'election policy vote']
    test_docs = ['rocket team', 'policy orbit']

조건:

- 학습 문서에는 fit_transform, 테스트 문서에는 transform을 사용하세요.
- X_train, X_test의 shape와 학습 어휘를 출력하세요.
- 테스트 문서의 단어로 다시 fit하면 왜 데이터 누수가 되는지 설명하세요.
- 테스트에만 있는 단어를 하나 추가해 변환 결과를 확인하세요.

## 문제 5. 최빈 클래스 기준선과 Naive Bayes 비교

문제 4의 문서에 아래 정답을 붙여 DummyClassifier와 MultinomialNB를 비교하세요.

    y_train = np.array([0, 1, 2])
    y_test = np.array([0, 2])

조건:

- DummyClassifier의 most_frequent 전략과 MultinomialNB를 각각 학습하세요.
- 두 모델의 accuracy와 macro F1을 출력하세요.
- 최빈 클래스 기준선이 필요한 이유를 작성하세요.
- 매우 작은 데이터의 점수 하나로 모델 우열을 단정하기 어려운 이유를 설명하세요.

## 문제 6. 주요 단어와 오분류 문서 읽기

학습한 MultinomialNB의 feature_log_prob_와 vectorizer.get_feature_names_out()으로 클래스별 주요 단어를 찾으세요.

조건:

- 클래스마다 로그확률이 큰 단어 3개를 출력하세요.
- argsort를 사용해 큰 값의 feature 인덱스를 찾으세요.
- 테스트 정답과 예측이 다른 행의 문서, 정답, 예측, 확률을 함께 출력하세요.
- 주요 단어가 인과관계나 완전한 설명과 같지 않은 이유를 작성하세요.

# 2. LSTM 순차 텍스트 분류

이 절에서는 토큰 ID가 임베딩과 LSTM을 지나 클래스 로짓이 되는 흐름을 확인합니다. B는 배치 문서 수, L은 패딩된 최대 토큰 길이, E는 임베딩 차원, H는 은닉 상태 차원입니다.

## 문제 7. 학습 문서만 사용해 PAD·OOV 사전 만들기

다음 문장으로 토큰 ID 사전을 만드세요.

    train_sentences = [
        'movie was very good',
        'movie was very boring',
        'acting was good',
    ]

조건:

- 소문자화와 공백 분리로 토큰화하세요.
- Counter로 빈도를 세고 PAD는 0, OOV는 1로 먼저 등록하세요.
- 나머지 단어에는 2부터 ID를 부여하세요.
- ID 숫자의 크기가 단어 의미나 중요도 순서는 아니라는 점을 주석으로 작성하세요.

## 문제 8. OOV 처리와 post-padding 구현

문제 7의 word_to_id를 사용해 아래 문장을 최대 길이 6의 토큰 ID 텐서로 바꾸세요.

    new_sentences = ['movie was good', 'unknown word was amazing']

조건:

- 학습 사전에 없는 단어는 OOV ID로 바꾸세요.
- 길이가 6보다 길면 뒤를 자르고, 짧으면 오른쪽에 PAD를 붙이세요.
- 결과 dtype을 torch.long, shape를 (문장 수, 6)으로 만드세요.
- 각 문장의 실제 길이와 PAD를 포함한 길이를 출력하세요.

## 문제 9. TensorDataset과 DataLoader의 배치 확인

문제 8의 ID 텐서와 아래 정답을 DataLoader로 묶으세요.

    labels = torch.tensor([1, 0], dtype=torch.long)

조건:

- TensorDataset과 DataLoader를 만들고 batch_size=2로 설정하세요.
- 첫 배치 입력 shape와 정답 shape를 출력하세요.
- B가 무엇을 뜻하는지, 마지막 배치는 BATCH_SIZE보다 작을 수 있는 이유를 작성하세요.
- 학습 loader에 shuffle=True를 주는 이유를 설명하세요.

## 문제 10. nn.Embedding의 행 조회와 shape

문제 8의 token_ids를 입력으로 사용할 nn.Embedding을 만드세요.

조건:

- nn.Embedding(len(word_to_id), 4, padding_idx=0)를 만드세요.
- 입력 (B, L)가 출력 (B, L, E)로 바뀌는 것을 확인하세요.
- 토큰 ID 하나가 임베딩 표의 어느 방향을 선택하는지 설명하세요.
- padding_idx=0이 PAD 위치를 텐서에서 삭제하는 기능이 아닌 이유를 작성하세요.

## 문제 11. 실제 길이와 pack_padded_sequence

PAD ID가 0인 입력에서 문장별 실제 길이를 계산하고 PackedSequence를 만드세요.

조건:

- token_ids.ne(0).sum(dim=1)으로 lengths를 구하고 clamp(min=1)를 적용하세요.
- 임베딩 결과와 lengths를 pack_padded_sequence에 전달하세요.
- batch_first=True와 enforce_sorted=False의 의미를 주석으로 설명하세요.
- PackedSequence가 단순한 1차원 텐서가 아니라 PAD를 건너뛰도록 만든 입력 표현이라는 점을 작성하세요.

## 문제 12. Embedding → LSTM → Linear 분류기

입력 (B, L)의 토큰 ID를 받아 두 감성 클래스의 로짓 (B, 2)을 반환하는 ReviewLSTMClassifier를 작성하세요.

조건:

- __init__에서 embedding, lstm, classifier를 정의하세요.
- embedding_dim=4, hidden_dim=8, num_classes=2를 사용하세요.
- forward에서 실제 길이를 계산하고 packed input을 LSTM에 넣으세요.
- 마지막 LSTM 층의 hidden[-1]을 문장 벡터로 사용하세요.
- hidden_dim은 은닉층 개수가 아니라 문장 요약 벡터의 길이라는 점을 주석으로 작성하세요.

## 문제 13. 순전파의 shape 추적

문제 12의 모델에 token_ids를 넣고 각 단계의 shape를 확인하세요.

조건:

- 입력 ID, 임베딩 결과, 문장 벡터, logits의 shape를 출력하세요.
- (B, L) → (B, L, E) → (B, H) → (B, C) 흐름을 작성하세요.
- logits의 값이 바로 확률이 아닌 이유를 설명하세요.
- model(token_ids)가 내부적으로 forward를 호출한다는 점을 주석으로 남기세요.

## 문제 14. CrossEntropyLoss의 입력과 정답 형식

문제 13의 logits와 labels를 CrossEntropyLoss에 넣어 loss를 계산하세요.

조건:

- logits shape가 (B, 2), 정답 shape가 (B,)이고 dtype이 torch.long인지 확인하세요.
- nn.CrossEntropyLoss로 loss를 계산하세요.
- forward의 마지막에 softmax를 먼저 적용하면 안 되는 이유를 작성하세요.
- argmax(dim=1)로 예측 ID를 만들고 정답과 비교하세요.

## 문제 15. 한 번의 학습 step과 역전파

Adam optimizer를 만들고 모델을 한 번 학습시키세요.

조건:

- zero_grad → forward → loss → backward → step 순서로 구현하세요.
- backward 뒤 classifier.weight.grad의 shape와 None 여부를 출력하세요.
- step 전후 임베딩의 good 토큰 행을 비교하세요.
- backward는 gradient를 계산하고 step은 실제 가중치를 수정한다는 점을 설명하세요.

## LSTM 학습·검증 심화

문제 16에서는 학습과 평가 코드를 분리하고, validation loss를 기준으로 가중치를 저장·복원하는 흐름을 완성합니다.

## 문제 16. 학습·검증 루프와 최적 가중치 저장

train_one_epoch와 evaluate 함수를 작성하고, 여러 epoch에서 validation loss가 가장 작을 때의 가중치를 저장하세요.

조건:

- 학습에는 model.train(), backward(), optimizer.step()을 사용하세요.
- 평가는 model.eval()과 torch.no_grad()에서 평균 loss와 macro F1을 구하세요.
- best_validation_loss가 개선될 때 copy.deepcopy(model.state_dict())를 저장하세요.
- 마지막에 load_state_dict로 최적 상태를 복원하세요.
- train loss는 줄었지만 validation loss가 오른 epoch를 과적합 후보로 찾으세요.

# 3. 멀티레이블 텍스트 분류

멀티레이블 문서는 여러 정답을 동시에 가질 수 있습니다. 따라서 클래스 하나만 고르는 softmax 분류와 달리 레이블별 독립 확률과 임계값이 필요합니다.

## 문제 17. multi-hot 정답 행렬 만들기

다음 장르 목록을 MultiLabelBinarizer로 변환하세요.

    label_lists = [
        ('판타지', '모험'),
        ('로맨스',),
        ('판타지',),
        ('모험', '로맨스'),
    ]
    label_order = ['판타지', '모험', '로맨스']

조건:

- classes=label_order로 MultiLabelBinarizer를 만들고 multi-hot 행렬 Y를 생성하세요.
- Y의 shape와 각 열이 뜻하는 레이블을 출력하세요.
- ('판타지', '모험')이 왜 클래스 ID 하나가 아니라 [1, 1, 0]인지 설명하세요.
- inverse_transform으로 multi-hot 행을 다시 레이블 목록으로 복원하세요.

## 문제 18. 문장·정답 인덱스 정렬 검증

아래 문장과 문제 17의 Y를 같은 인덱스로 학습·테스트로 나누세요.

    texts = [
        '마법 왕국 모험',
        '두 사람의 사랑',
        '용과 마법',
        '여행과 사랑',
    ]

조건:

- train_indices와 test_indices를 만들고 train_texts, test_texts를 만드세요.
- y_train은 Y[train_indices], y_test는 Y[test_indices]로 만드세요.
- X를 만든 뒤 X_train.shape[0] == y_train.shape[0]과 X_test.shape[0] == y_test.shape[0]을 assert로 확인하세요.
- y_test에 train_indices를 실수로 쓰면 어떤 오류가 나는지 설명하세요.

## 문제 19. One-vs-Rest Logistic Regression 학습

문제 17의 학습·테스트 문장을 TF-IDF로 바꾸고 OneVsRestClassifier를 학습하세요.

조건:

- ngram_range=(1, 2) TF-IDF를 학습 문장에만 fit하세요.
- LogisticRegression을 OneVsRestClassifier로 감싸 multi-hot y_train을 학습하세요.
- predict_proba 결과 test_probabilities의 shape를 출력하세요.
- 각 레이블에 이진 분류기 하나가 학습된다는 뜻을 주석으로 작성하세요.

## 문제 20. 독립 확률과 threshold 예측

문제 18의 test_probabilities를 임계값 0.3, 0.5, 0.7에서 각각 0/1 예측으로 바꾸세요.

조건:

- probabilities >= threshold 비교로 예측 행렬을 만드세요.
- 각 threshold에서 문장당 예측 레이블 수 평균을 출력하세요.
- 한 행의 확률 합이 꼭 1일 필요가 없는 이유를 설명하세요.
- softmax 단일 레이블 분류와 sigmoid 기반 멀티레이블 분류의 차이를 작성하세요.

## 문제 21. micro F1, macro F1, Hamming loss 비교

문제 19의 예측을 y_test와 비교해 평가 지표를 계산하세요.

조건:

- f1_score의 average='micro'와 average='macro'를 각각 구하세요.
- hamming_loss를 구하세요.
- zero_division=0이 필요한 상황을 설명하세요.
- 레이블 불균형이 있을 때 macro F1을 함께 보는 이유를 작성하세요.

## 멀티레이블 기준선·임계값 선택

문제 22에서는 학습 레이블의 출현 비율만 반복하는 기준선과, validation 기반 임계값 선택의 필요성을 확인합니다.

## 문제 22. 레이블 비율 기준선과 임계값 선택

학습 레이블의 출현 비율만 사용해 멀티레이블 기준선을 만들고, threshold 선택 위치를 설명하세요.

조건:

- y_train.mean(axis=0)으로 label_prevalence를 구하세요.
- 0.5 이상 레이블을 모든 테스트 행에 반복하는 baseline_predictions를 만드세요.
- 모델 결과와 기준선의 micro F1, macro F1, Hamming loss를 비교하세요.
- 실제 프로젝트에서 test set이 아니라 validation set으로 threshold를 고르는 이유를 작성하세요.

# 4. TextCNN 텍스트 분류

TextCNN은 여러 크기의 합성곱 커널로 문장 안의 짧은 지역 표현을 찾고, Global Max Pooling으로 각 표현의 가장 강한 반응을 요약합니다.

## 문제 23. TextCNN용 토큰 ID와 입력 축

아래 짧은 감성 문장을 학습용 사전으로 인코딩하고, TextCNN 입력 shape를 확인하세요.

    reviews = [
        '영화 정말 재미 있다',
        '연출 매우 지루 하다',
        '배우 연기 좋다',
        '결말 너무 나쁘다',
    ]

조건:

- PAD=0, OOV=1을 포함한 사전을 만들고 최대 길이 6으로 post-padding 하세요.
- 입력 ID 텐서 shape (B, L)를 출력하세요.
- 임베딩 차원 E=4일 때 임베딩 출력 shape (B, L, E)를 확인하세요.
- CNN 전에 토큰화·PAD·OOV 규칙을 고정해야 하는 이유를 작성하세요.

## 문제 24. Conv1d 입력 축과 커널 출력 길이

문제 23의 임베딩 결과를 Conv1d에 넣기 위해 축을 바꾸고, 커널 크기 2와 3의 출력 shape를 확인하세요.

조건:

- embedded.transpose(1, 2)로 (B, E, L) 입력을 만드세요.
- nn.Conv1d(in_channels=4, out_channels=3, kernel_size=2)를 적용하세요.
- kernel_size=2와 3에서 위치 축 길이가 어떻게 달라지는지 출력하세요.
- Conv1d가 (B, L, E)가 아니라 (B, E, L)를 받는 이유를 주석으로 작성하세요.

## 문제 25. Global Max Pooling으로 지역 특징 요약

문제 24의 feature map에서 토큰 위치 축의 최댓값만 남기세요.

조건:

- ReLU 뒤 max(dim=2)를 적용해 필터마다 값 하나를 얻으세요.
- pooling 전후 shape를 출력하세요.
- Global Max Pooling이 문장 어디엔가 특정 표현이 강하게 있었는지를 나타내는 이유를 설명하세요.
- max pooling이 위치 정보를 잃는 한계를 한 문장으로 작성하세요.

## 문제 26. 병렬 커널 TextCNN 클래스 작성

커널 크기 2, 3, 4를 병렬로 사용하는 TextCNNClassifier를 작성하세요.

조건:

- embedding, ModuleList의 Conv1d 계층, dropout, classifier를 정의하세요.
- 각 Conv1d의 out_channels는 4로 두세요.
- forward에서 임베딩 → transpose → convolution과 ReLU → global max pooling → concat → dropout → classifier 순서로 구현하세요.
- 입력 (B, L)에서 최종 logits (B, 2)가 나오는지 확인하세요.
- concat 뒤 feature 길이가 4 × 3인 이유를 주석으로 작성하세요.

## 문제 27. TextCNN 학습과 train·eval 모드

문제 25의 TextCNN을 감성 정답으로 학습하고 평가하세요.

조건:

- CrossEntropyLoss와 Adam으로 여러 epoch 학습하세요.
- 학습 단계에서는 model.train(), 평가 단계에서는 model.eval()과 torch.no_grad()를 사용하세요.
- dropout이 train과 eval 모드에서 다르게 동작하는 이유를 설명하세요.
- 최빈 클래스 기준선과 accuracy 또는 macro F1을 비교하세요.

## 문제 28. 서로 다른 문장이 같은 OOV 입력이 되는 한계

학습 사전에 없는 아래 두 문장을 같은 최대 길이로 인코딩하고 예측을 비교하세요.

    oov_texts = ['웅장한 촬영미', '황홀한 연출미']

조건:

- 두 문장의 토큰 ID 행이 같은지 출력하세요.
- TextCNN의 logits 또는 softmax 확률이 같은지 확인하세요.
- 같은 OOV/PAD ID 행이면 이후 모델이 원문 차이를 구분할 수 없는 이유를 설명하세요.
- 서브워드 토큰화나 더 큰 사전이 이 한계를 어떻게 완화할 수 있는지 작성하세요.

## 문제 29. 종합: 분류 모델 선택 보고서

아래 네 상황에서 첫 기준선과 비교 후보를 고르고, 평가 지표와 가장 큰 위험을 표로 작성하세요.

1. 근거 단어를 보여줘야 하는 고객 문의 분류
2. 문장 내 순서와 긴 문맥이 중요한 뉴스 주제 분류
3. 한 문서에 여러 태그가 붙는 영화 장르 분류
4. 짧은 감성 표현이 반복되는 대량 리뷰 분류

조건:

- Naive Bayes, LSTM, One-vs-Rest, TextCNN 중 적절한 방법을 연결하세요.
- 각 선택의 입력 표현, 출력 형식, 정답 형식, 주요 평가 지표를 함께 쓰세요.
- 데이터 누수, 클래스 불균형, OOV, 과적합 중 상황별로 가장 큰 위험을 하나 이상 적으세요.
- 가장 복잡한 모델을 처음부터 쓰지 않는 실무적 이유를 2문장으로 정리하세요.